# RDR2 Sentiment Analysis — API Connectivity Test

This notebook verifies that all external API credentials load correctly from `.env` and that each data source used in the project is reachable, **before** running the full data collection pipeline.

It does **not** print raw key values — only masked previews and boolean checks — so this notebook is safe to keep in version control or share without exposing credentials.

**APIs tested:**
1. Steam Web API (current player count)
2. Steam Reviews (public endpoint, no key required)
3. SteamSpy (public endpoint, no key required)
4. YouTube Data API v3 (search)
5. IGDB API (via Twitch OAuth)
6. Google Trends (via `pytrends`, no key required)


## Setup
Install/confirm `python-dotenv`, used to load credentials from `.env` without hardcoding them in the notebook.

In [ ]:
pip install python-dotenv --break-system-packages

## Load credentials from `.env`
All keys are loaded as environment variables — never typed directly into this notebook.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

STEAM_API_KEY = os.getenv("STEAM_API_KEY")
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")
CLIENT_ID = os.getenv("IGDB_CLIENT_ID")
CLIENT_SECRET = os.getenv("IGDB_CLIENT_SECRET")

## Verify keys loaded (masked)
Confirms each credential was read successfully from `.env`, without ever printing the full value. Shows only the first 4 characters plus length, enough to sanity-check the right key loaded without exposing it.

In [ ]:
def mask(value, visible=4):
    """Return a masked preview of a secret: first `visible` chars + length, or None if missing."""
    if not value:
        return None
    return f"{value[:visible]}{'*' * max(len(value) - visible, 0)} (len={len(value)})"

print("STEAM_API_KEY:   ", mask(STEAM_API_KEY))
print("YOUTUBE_API_KEY: ", mask(YOUTUBE_API_KEY))
print("IGDB_CLIENT_ID:  ", mask(CLIENT_ID))
print("IGDB_CLIENT_SECRET:", mask(CLIENT_SECRET))

assert STEAM_API_KEY, "Missing STEAM_API_KEY in .env"
assert YOUTUBE_API_KEY, "Missing YOUTUBE_API_KEY in .env"
assert CLIENT_ID, "Missing IGDB_CLIENT_ID in .env"
assert CLIENT_SECRET, "Missing IGDB_CLIENT_SECRET in .env"
print("\nAll required keys present.")

## 1. Steam Web API
Sanity check: fetch current player counts for RDR2 and GTA V. This specific endpoint doesn't actually require a key, but confirms basic connectivity to Steam's API before the keyed calls.

In [ ]:
import requests

def test_steam_player_count(appid):
    """Fetch current player count for a given Steam appid."""
    url = f"https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/?appid={appid}"
    r = requests.get(url)
    print(r.status_code, r.json())

test_steam_player_count(1174180)  # RDR2
test_steam_player_count(271590)   # GTA V

## 2. Steam Reviews
Public endpoint, no key required. Confirms the reviews endpoint used throughout the main data collection pipeline is reachable and returns expected fields.

In [ ]:
def test_steam_reviews(appid):
    """Fetch a small sample of reviews to confirm the endpoint and response shape."""
    url = f"https://store.steampowered.com/appreviews/{appid}?json=1&num_per_page=5"
    r = requests.get(url)
    data = r.json()
    print(data["query_summary"])
    print(data["reviews"][0]["review"][:200])  # preview first review

test_steam_reviews(1174180)  # RDR2

## 3. SteamSpy
Public endpoint, no key required. Used for supplementary game metadata (owners estimate, tags, pricing).

In [ ]:
def test_steamspy(appid):
    """Fetch SteamSpy metadata for a given appid."""
    url = f"https://steamspy.com/api.php?request=appdetails&appid={appid}"
    r = requests.get(url)
    print(r.json())

test_steamspy(1174180)  # RDR2

## 4. YouTube Data API v3
Requires `YOUTUBE_API_KEY`. Confirms search endpoint returns results before the full comment-collection pipeline runs.

In [ ]:
def test_youtube_search(query):
    """Run a small search query against the YouTube Data API to confirm the key works."""
    url = "https://www.googleapis.com/youtube/v3/search"
    params = {
        "part": "snippet",
        "q": query,
        "maxResults": 3,
        "key": YOUTUBE_API_KEY
    }
    r = requests.get(url, params=params)
    print(r.status_code)
    for item in r.json().get("items", []):
        print(item["snippet"]["title"])

test_youtube_search("RDR2 Arthur Morgan death scene")

## 5. IGDB API (via Twitch OAuth)
Requires `IGDB_CLIENT_ID` and `IGDB_CLIENT_SECRET`. First fetches an OAuth token from Twitch, then queries the IGDB `characters` endpoint to confirm both the token exchange and the API call work end-to-end.

In [ ]:
def get_igdb_token():
    """Exchange IGDB/Twitch client credentials for a short-lived OAuth access token."""
    url = "https://id.twitch.tv/oauth2/token"
    params = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "client_credentials"
    }
    r = requests.post(url, params=params)
    return r.json()["access_token"]

token = get_igdb_token()

def test_igdb_characters():
    """Search IGDB for a known character to confirm the API responds correctly."""
    url = "https://api.igdb.com/v4/characters"
    headers = {
        "Client-ID": CLIENT_ID,
        "Authorization": f"Bearer {token}"
    }
    body = 'search "Arthur Morgan"; fields name,games;'
    r = requests.post(url, headers=headers, data=body)
    print(r.json())

test_igdb_characters()

## 6. Google Trends (via `pytrends`)
No key required. **Note:** this endpoint is aggressively rate-limited by Google and returns `429 Too Many Requests` if called too frequently — this is expected and not a credentials issue. Re-run after waiting a few minutes if it fails here.

In [ ]:
# pip install pytrends --break-system-packages
from pytrends.request import TrendReq

pytrends = TrendReq(hl="en-US", tz=360)
pytrends.build_payload(["Red Dead Redemption 2"], timeframe="2019-01-01 2026-07-01")
df = pytrends.interest_over_time()
print(df.head())

## Summary
If every cell above ran without error (aside from a possible Google Trends rate-limit), all credentials are valid and the main data collection notebooks are safe to run.